In [3]:
#loading metacells of rna and atac data
import anndata
from anndata import AnnData
import pandas as pd
import scanpy as sc

sc_rna = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/SEAcells/SEACell_summarized_RNA.h5ad")  
sc_atac = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/SEAcells/SEACell_summarized_ATAC.h5ad")
rna_mapping = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/SEAcells/RNA_SEACell_mapping.csv", sep='\t', index_col=0)

gene_peaks_10kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Rmd_n_notebook/gene_peak_assignments_10kb.csv")
gene_peaks_20kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Rmd_n_notebook/gene_peak_assignments_20kb.csv")
gene_peaks_50kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Rmd_n_notebook/gene_peak_assignments_50kb.csv")
gene_peaks_100kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Rmd_n_notebook/gene_peak_assignments_100kb.csv")

IORegistryError: No read method registered for IOSpec(encoding_type='null', encoding_version='0.1.0') from <class 'h5py._hl.dataset.Dataset'>. You may need to update your installation of anndata.

In [ ]:
#function for normalizing the aggregated data by total counts per SEACell to get relative accessibility/expression values (compositional normalization)
def compositional_normalize(df):
    row_sums = df.sum(axis=1)
    return df.div(row_sums, axis=0)

sc_rna_norm = compositional_normalize(sc_rna)
sc_atac_norm = compositional_normalize(sc_atac)

##Von hier anpassen zu sc_atac/sc_rna!!


In [ ]:
#check dimensions of the data
print(sc_atac_norm.X.shape)
print(sc_rna_norm.X.shape)

print(sc_atac_norm.obs_names)
print(sc_rna_norm.obs_names)

In [ ]:
#Reindex the dataframes to ensure they are in the same order
sc_atac_norm = sc_atac_norm[sc_rna_norm.obs_names, :]
sc_rna_norm = sc_rna_norm[sc_atac_norm.obs_names, :]
print(sc_atac_norm.obs_names.equals(sc_rna_norm.obs_names))  # Should return True

sc_rna_norm.shape

In [ ]:
#Substract the genes that are present in the gene_peaks_10kb dataframe from the sc_rna_norm dataframe, to only keep the genes that have peaks assigned to them
genes_with_peaks_10kb = gene_peaks_10kb["gene_id"].tolist()
sc_rna_norm = sc_rna_norm[:, genes_with_peaks_10kb] 
print(sc_rna_norm.shape)

In [ ]:
print(sc_atac_norm.X)
print(sc_rna_norm.X)

In [ ]:
#taking the minimum non-zero value in the sc_rna_norm matrix to add it to all values before log transformation to avoid taking log of zero
import numpy as np
non_zero_mask = (sc_rna_norm.X > 0)
epsilon_rna = np.min(sc_rna_norm.X[non_zero_mask])

In [ ]:
#Plotting the distribution of the value of the genes across all cell types in a boxplot for RNA data
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

P_rna_scaled = np.log10((sc_rna_norm.X+ epsilon_rna)*10000000) 

# RNA counts x samples
P_rna_scaled = P_rna_scaled.T.astype(float)

# convert to long format for seaborn
P_rna_scaled_plot = pd.DataFrame(P_rna_scaled, columns=sc_rna_norm.obs_names).melt(
	var_name="celltype",
	value_name="log_cpm"
)
#Plotting the distribution of values of the genes across all cell types in a density plot for RNA data
plt.figure(figsize=(10, 5))
sns.kdeplot(data=P_rna_scaled_plot, x="log_cpm", common_norm=False, fill=True, alpha=0.5)
plt.xlabel("log10((RNA + min(RNA)) * 10000000)")
plt.title("Density of log10((RNA + min(RNA)) * 10000000) across cell type")
plt.tight_layout()
plt.show()

In [ ]:
#taking the minimum non-zero value in the sc_atac_norm matrix to add it to all values before log transformation to avoid taking log of zero
non_zero_mask = (sc_atac_norm.X > 0)
epsilon_at= np.min(sc_atac_norm.X[non_zero_mask])

In [ ]:
#Plotting the distribution of the value of the peaks across all cell types in a boxplot for ATAC data
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

P_atac_scaled = np.log10((sc_atac_norm.X+ epsilon_at)*10000000) 

# ATAC counts x samples
P_atac_scaled = P_atac_scaled.T.astype(float)

# convert to long format for seaborn
P_atac_scaled_plot = pd.DataFrame(P_atac_scaled, columns=sc_atac_norm.obs_names).melt(
	var_name="celltype",
	value_name="log_cpm"
)
#Plotting the distribution of values of the peaks across all cell types in a density plot for ATAC data
plt.figure(figsize=(10, 5))
sns.kdeplot(data=P_atac_scaled_plot, x="log_cpm", common_norm=False, fill=True, alpha=0.5)
plt.xlabel("log10((ATAC + min(ATAC)) * 10000000)")
plt.title("Density of log10((ATAC + min(ATAC)) * 10000000) across cell type")
plt.tight_layout()
plt.show()

In [ ]:
# dictionary to save peak-wise correlation results per gene in all windows
from pathlib import Path
import sys

repo_root = Path("/home/fgsasse_lrs_1/Downloads/BA")
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from your_package.correlation import (
    DEFAULT_WINDOW_LABELS,
    add_correlation_categories,
    aggregate_correlation_categories,
    build_correlation_dataframe,
    compute_all_window_peak_correlations,
)

window_assignments = {
    "10kb": gene_peaks_10kb,
    "20kb": gene_peaks_20kb,
    "50kb": gene_peaks_50kb,
    "100kb": gene_peaks_100kb,
}

all_window_results = compute_all_window_peak_correlations(
    atac_data=sc_atac_norm,
    rna_data=sc_rna_norm,
    window_assignments=window_assignments,
)

cor_res_df = build_correlation_dataframe(
    all_window_results,
    window_labels=DEFAULT_WINDOW_LABELS,
)

#save the correlation results as a csv file
cor_res_df.to_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Rmd_n_notebook/gene_peak_correlation_sc_results.csv", index=False)

cor_res_df = add_correlation_categories(cor_res_df)
agg_cor_df = aggregate_correlation_categories(cor_res_df)


print(cor_res_df.shape)
print(cor_res_df.head())
print(agg_cor_df.head())

In [ ]:
# dictionary to save peak-wise correlation results per gene in 20kb window
import ast
import numpy as np
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

# 1. Compute correlations 
gene_peak_20kb_cor_results = {}

for gene_id in gene_peaks_20kb["gene_id"]:
    assigned_peaks_raw = gene_peaks_20kb.loc[
        gene_peaks_20kb["gene_id"] == gene_id, "assigned_peaks"
    ].iloc[0]

    if isinstance(assigned_peaks_raw, str):
        assigned_peaks = ast.literal_eval(assigned_peaks_raw)
    else:
        assigned_peaks = assigned_peaks_raw

    assigned_peaks = [p for p in assigned_peaks if p in sc_atac_norm.var_names]

    peak_corrs = {}
    if len(assigned_peaks) == 0:
        gene_peak_20kb_cor_results[gene_id] = peak_corrs
        continue

    gene_expression = np.asarray(sc_rna_norm[:, gene_id].X).ravel()

    for peak_id in assigned_peaks:
        peak_accessibility = np.asarray(sc_atac_norm[:, peak_id].X).ravel()

        if np.std(gene_expression) == 0 or np.std(peak_accessibility) == 0:
            peak_corrs[peak_id] = {"corr": np.nan, "pval": np.nan, "padj": np.nan}
        else:
            corr, pval = pearsonr(gene_expression, peak_accessibility)
            peak_corrs[peak_id] = {"corr": corr, "pval": pval, "padj": np.nan}  # padj filled below

    gene_peak_20kb_cor_results[gene_id] = peak_corrs


#2. Collect all valid p-values across every peak–gene pair 
pair_index = []   # keeps track of (gene_id, peak_id) so we can write padj back
pvals_all  = []

for gene_id, peak_corrs in gene_peak_20kb_cor_results.items():
    for peak_id, stats in peak_corrs.items():
        if not np.isnan(stats["pval"]):          # skip NaN (constant vectors)
            pair_index.append((gene_id, peak_id))
            pvals_all.append(stats["pval"])


# 3. Apply Benjamini-Hochberg FDR correction 
if len(pvals_all) > 0:
    _, padj_all, _, _ = multipletests(pvals_all, method="fdr_bh")

    # write adjusted p-values back into the results dictionary
    for (gene_id, peak_id), padj in zip(pair_index, padj_all):
        gene_peak_20kb_cor_results[gene_id][peak_id]["padj"] = padj

In [ ]:
# dictionary to save peak-wise correlation results per gene in 50kb window
import ast
import numpy as np
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

# 1. Compute correlations 
gene_peak_50kb_cor_results = {}

for gene_id in gene_peaks_50kb["gene_id"]:
    assigned_peaks_raw = gene_peaks_50kb.loc[
        gene_peaks_50kb["gene_id"] == gene_id, "assigned_peaks"
    ].iloc[0]

    if isinstance(assigned_peaks_raw, str):
        assigned_peaks = ast.literal_eval(assigned_peaks_raw)
    else:
        assigned_peaks = assigned_peaks_raw

    assigned_peaks = [p for p in assigned_peaks if p in sc_atac_norm.var_names]

    peak_corrs = {}
    if len(assigned_peaks) == 0:
        gene_peak_50kb_cor_results[gene_id] = peak_corrs
        continue

    gene_expression = np.asarray(sc_rna_norm[:, gene_id].X).ravel()

    for peak_id in assigned_peaks:
        peak_accessibility = np.asarray(sc_atac_norm[:, peak_id].X).ravel()

        if np.std(gene_expression) == 0 or np.std(peak_accessibility) == 0:
            peak_corrs[peak_id] = {"corr": np.nan, "pval": np.nan, "padj": np.nan}
        else:
            corr, pval = pearsonr(gene_expression, peak_accessibility)
            peak_corrs[peak_id] = {"corr": corr, "pval": pval, "padj": np.nan}  # padj filled below

    gene_peak_50kb_cor_results[gene_id] = peak_corrs


#2. Collect all valid p-values across every peak–gene pair 
pair_index = []   # keeps track of (gene_id, peak_id) so we can write padj back
pvals_all  = []

for gene_id, peak_corrs in gene_peak_50kb_cor_results.items():
    for peak_id, stats in peak_corrs.items():
        if not np.isnan(stats["pval"]):          # skip NaN (constant vectors)
            pair_index.append((gene_id, peak_id))
            pvals_all.append(stats["pval"])


# 3. Apply Benjamini-Hochberg FDR correction 
if len(pvals_all) > 0:
    _, padj_all, _, _ = multipletests(pvals_all, method="fdr_bh")

    # write adjusted p-values back into the results dictionary
    for (gene_id, peak_id), padj in zip(pair_index, padj_all):
        gene_peak_50kb_cor_results[gene_id][peak_id]["padj"] = padj

In [ ]:
# dictionary to save peak-wise correlation results per gene in 100kb window
import ast
import numpy as np
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

# 1. Compute correlations 
gene_peak_100kb_cor_results = {}

for gene_id in gene_peaks_100kb["gene_id"]:
    assigned_peaks_raw = gene_peaks_100kb.loc[
        gene_peaks_100kb["gene_id"] == gene_id, "assigned_peaks"
    ].iloc[0]

    if isinstance(assigned_peaks_raw, str):
        assigned_peaks = ast.literal_eval(assigned_peaks_raw)
    else:
        assigned_peaks = assigned_peaks_raw

    assigned_peaks = [p for p in assigned_peaks if p in sc_atac_norm.var_names]

    peak_corrs = {}
    if len(assigned_peaks) == 0:
        gene_peak_100kb_cor_results[gene_id] = peak_corrs
        continue

    gene_expression = np.asarray(sc_rna_norm[:, gene_id].X).ravel()

    for peak_id in assigned_peaks:
        peak_accessibility = np.asarray(sc_atac_norm[:, peak_id].X).ravel()

        if np.std(gene_expression) == 0 or np.std(peak_accessibility) == 0:
            peak_corrs[peak_id] = {"corr": np.nan, "pval": np.nan, "padj": np.nan}
        else:
            corr, pval = pearsonr(gene_expression, peak_accessibility)
            peak_corrs[peak_id] = {"corr": corr, "pval": pval, "padj": np.nan}  # padj filled below

    gene_peak_100kb_cor_results[gene_id] = peak_corrs


#2. Collect all valid p-values across every peak–gene pair 
pair_index = []   # keeps track of (gene_id, peak_id) so we can write padj back
pvals_all  = []

for gene_id, peak_corrs in gene_peak_100kb_cor_results.items():
    for peak_id, stats in peak_corrs.items():
        if not np.isnan(stats["pval"]):          # skip NaN (constant vectors)
            pair_index.append((gene_id, peak_id))
            pvals_all.append(stats["pval"])


# 3. Apply Benjamini-Hochberg FDR correction 
if len(pvals_all) > 0:
    _, padj_all, _, _ = multipletests(pvals_all, method="fdr_bh")

    # write adjusted p-values back into the results dictionary
    for (gene_id, peak_id), padj in zip(pair_index, padj_all):
        gene_peak_100kb_cor_results[gene_id][peak_id]["padj"] = padj

In [ ]:
# sanity check: print the shape of the gene expression vector for one gene and the accessibility vector for one peak assigned to that gene
for gene_id in gene_peaks_20kb["gene_id"]:
    assigned_peaks_raw = gene_peaks_20kb.loc[
        gene_peaks_20kb["gene_id"] == gene_id, "assigned_peaks"
    ].iloc[0]
x = sc_rna_norm[:, gene_id].X
print(x.shape)

In [ ]:
# Build one tidy table across all windows and save as CSV for downstream analysis
import numpy as np
import pandas as pd

window_assignments = {
    "10kb":  gene_peak_10kb_cor_results,
    "20kb":  gene_peak_20kb_cor_results,
    "50kb":  gene_peak_50kb_cor_results,
    "100kb": gene_peak_100kb_cor_results,
}

all_window_results = []

for window_label, cor_results in window_assignments.items():
    for gene, peaks in cor_results.items():
        for peak, stats in peaks.items():

            # guard: skip NaN entries (constant vectors)
            if not isinstance(stats, dict):
                continue

            corr = stats.get("corr", np.nan)
            pval = stats.get("pval", np.nan)
            padj = stats.get("padj", np.nan)

            # keep only rows with finite, valid values
            if np.isfinite(corr) and np.isfinite(pval) and 0 < pval <= 1:
                all_window_results.append(
                    {
                        "window":           window_label,
                        "gene":             gene,
                        "peak":             peak,
                        "correlation":      corr,
                        "pvalue":           pval,
                        "neglog10_pvalue":  -np.log10(pval),
                        "padj":             padj,
                        "neglog10_padj":    -np.log10(padj) if np.isfinite(padj) and padj > 0 else np.nan,
                    }
                )

cor_res_df = pd.DataFrame(all_window_results)

# make window an ordered categorical for correct plot ordering
cor_res_df["window"] = pd.Categorical(
    cor_res_df["window"],
    categories=["10kb", "20kb", "50kb", "100kb"],
    ordered=True
)

#save the correlation results as a csv file
cor_res_df.to_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Rmd_n_notebook/gene_peak_correlation_sc_results.csv", index=False)
print(cor_res_df.shape)
print(cor_res_df.head())